In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv('merged.csv', encoding='utf-8-sig')
print(f"시작: {len(df)}행")

시작: 52358행


### 1. regulation_type 인코딩
- 반덤핑 0 / 상계관세 1 / 세이프가드 2

In [25]:
reg_type_map = {'반덤핑': 0, '상계관세': 1, '세이프가드': 2}
df['regulation_type_encoded'] = df['regulation_type'].map(reg_type_map)
print(f"\nregulation_type 인코딩 결측: {df['regulation_type_encoded'].isna().sum()}건")


regulation_type 인코딩 결측: 0건


### 2. regulation_status 인코딩
- 규제중 1 / 조사중 0

In [26]:
reg_status_map = {'규제중': 1, '조사중': 0}
df['regulation_status_encoded'] = df['regulation_status'].map(reg_status_map)
print(f"regulation_status 인코딩 결측: {df['regulation_status_encoded'].isna().sum()}건")

regulation_status 인코딩 결측: 0건


### 3. duration_months 계산
- end_date 결측 -> -1
  - 그러나 조사중 자체가 의미있는 정보라서 그대로 유지 해야함
  - 제거하면 데이터 부족해짐...

In [27]:
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date']   = pd.to_datetime(df['end_date'],   errors='coerce')

df['duration_months'] = (
    (df['end_date'].dt.year  - df['start_date'].dt.year)  * 12 +
    (df['end_date'].dt.month - df['start_date'].dt.month)
)

In [28]:
df['duration_months'] = df['duration_months'].fillna(-1).astype(int)
print(f"duration_months 분포:\n{df['duration_months'].describe()}")

duration_months 분포:
count    52358.000000
mean        84.694774
std         52.957964
min         -1.000000
25%         60.000000
50%         60.000000
75%        132.000000
max        203.000000
Name: duration_months, dtype: float64


In [29]:
# 결측 개수
print(df['duration_months'].value_counts()[-1])

8321


### 4. 추가 인코딩(imposing_country / hs_4digit)

In [30]:
country_map = {'US': 0, 'CA': 1, 'GB': 2, 'TH': 3,
               'EU': 4, 'CN': 5, 'IN': 6}
df['imposing_country_encoded'] = df['imposing_country'].map(country_map)

In [31]:
hs_list = sorted(df['hs_4digit'].unique())
hs_map  = {hs: i for i, hs in enumerate(hs_list)}
df['hs_encoded'] = df['hs_4digit'].map(hs_map)

In [32]:
print(f"imposing_country 인코딩: {country_map}")
print(f"hs_4digit 인코딩: {hs_map}")
print(f"결측값 확인:")
print(f"  imposing_country_encoded: {df['imposing_country_encoded'].isna().sum()}건")
print(f"  hs_encoded: {df['hs_encoded'].isna().sum()}건")

imposing_country 인코딩: {'US': 0, 'CA': 1, 'GB': 2, 'TH': 3, 'EU': 4, 'CN': 5, 'IN': 6}
hs_4digit 인코딩: {np.int64(7202): 0, np.int64(7208): 1, np.int64(7209): 2, np.int64(7210): 3, np.int64(7211): 4, np.int64(7212): 5, np.int64(7213): 6, np.int64(7214): 7, np.int64(7215): 8, np.int64(7216): 9, np.int64(7217): 10, np.int64(7218): 11, np.int64(7219): 12, np.int64(7220): 13, np.int64(7222): 14, np.int64(7225): 15, np.int64(7226): 16, np.int64(7227): 17, np.int64(7228): 18, np.int64(7229): 19}
결측값 확인:
  imposing_country_encoded: 0건
  hs_encoded: 0건


### 5. dump_signal 생성
- 중앙값과 분위수 비교를 통해 dump_signal 임계값 판단 -> 뚜렷한 구분선이 없음 ...
- Recall은 높으면서 오탐률(FPR)은 낮은 임계값을 선택
  - price=0,  vol=-20  → 포착률 22.4%, 오탐률  5.3%
  - price=-5, vol=-10  → 포착률 39.3%, 오탐률 10.6% -> 선택

In [33]:
korea = df[df['is_korea_target'] == 1]
other = df[df['is_korea_target'] == 0]

chg_cols = ['price_chg_3m_new', 'price_chg_6m_new', 'price_chg_12m_new',
            'vol_chg_3m_new',   'vol_chg_6m_new',   'vol_chg_12m_new']

print("=== 한국 피소(1) 기술통계 ===")
print(korea[chg_cols].describe())

print("\n=== 비피소(0) 기술통계 ===")
print(other[chg_cols].describe())

print("\n=== 중앙값 비교 ===")
comparison = pd.DataFrame({
    '한국 피소(1)': korea[chg_cols].median(),
    '비피소(0)':    other[chg_cols].median(),
})
comparison['차이'] = comparison['한국 피소(1)'] - comparison['비피소(0)']
print(comparison)

print("\n=== 25th/75th percentile 비교 ===")
for col in chg_cols:
    k25, k75 = korea[col].quantile(0.25), korea[col].quantile(0.75)
    o25, o75 = other[col].quantile(0.25), other[col].quantile(0.75)
    print(f"\n{col}:")
    print(f"  한국 피소: 25th={k25:.1f}%  중앙={korea[col].median():.1f}%  75th={k75:.1f}%")
    print(f"  비피소:    25th={o25:.1f}%  중앙={other[col].median():.1f}%  75th={o75:.1f}%")

=== 한국 피소(1) 기술통계 ===
       price_chg_3m_new  price_chg_6m_new  price_chg_12m_new  vol_chg_3m_new  \
count      26024.000000      26024.000000       26024.000000    26024.000000   
mean           2.860636         -0.890294           5.951481        6.951179   
std           28.561067         33.685129          46.551896       60.814753   
min          -97.019649        -99.794920         -48.921996      -68.522823   
25%           -6.417658         -5.010455         -15.702817      -26.386738   
50%            3.563144          3.302918          -1.813951      -16.582117   
75%           14.687639         14.987430           7.777748       38.769353   
max           98.458512         80.078430         215.249829      265.641115   

       vol_chg_6m_new  vol_chg_12m_new  
count    26024.000000     26024.000000  
mean        42.655816       -27.900616  
std        184.156785        58.117301  
min        -86.988716       -98.980492  
25%        -36.364213       -55.703352  
50%        

In [34]:
# 여러 임계값 조합 테스트
price_thresholds = [0, -5, -10]
vol_thresholds   = [-10, -20, -30]

print("=== 임계값별 한국 피소 포착률 vs 오탐률 ===")
print(f"{'price_th':>10} {'vol_th':>8} {'포착률(한국피소중)':>18} {'오탐률(비피소중)':>16} {'정밀도':>8}")

for pt in price_thresholds:
    for vt in vol_thresholds:
        signal = (df['price_chg_12m_new'] < pt) & (df['vol_chg_12m_new'] < vt)

        korea_captured = signal[df['is_korea_target']==1].sum()
        korea_total    = (df['is_korea_target']==1).sum()
        other_captured = signal[df['is_korea_target']==0].sum()
        other_total    = (df['is_korea_target']==0).sum()

        recall    = korea_captured / korea_total * 100
        fpr       = other_captured / other_total * 100
        precision = korea_captured / signal.sum() * 100 if signal.sum() > 0 else 0

        print(f"{pt:>10} {vt:>8} {recall:>17.1f}% {fpr:>15.1f}% {precision:>7.1f}%")

=== 임계값별 한국 피소 포착률 vs 오탐률 ===
  price_th   vol_th         포착률(한국피소중)        오탐률(비피소중)      정밀도
         0      -10              39.3%            14.8%    73.4%
         0      -20              22.4%             5.3%    81.5%
         0      -30              22.4%             4.4%    83.9%
        -5      -10              39.3%            10.6%    79.4%
        -5      -20              22.4%             5.3%    81.5%
        -5      -30              22.4%             4.4%    83.9%
       -10      -10              28.3%             7.3%    80.1%
       -10      -20              11.4%             2.0%    85.5%
       -10      -30              11.4%             1.4%    89.7%


In [35]:
df['dump_signal'] = (
    (df['price_chg_12m_new'] < -5) &
    (df['vol_chg_12m_new']   < -10)
).astype(int)

In [36]:
print(f"\ndump_signal 분포:\n{df['dump_signal'].value_counts()}")
print(f"한국 피소 중 dump_signal=1: {df[df['is_korea_target']==1]['dump_signal'].sum()}건 "
      f"({df[df['is_korea_target']==1]['dump_signal'].mean()*100:.1f}%)")
print(f"비피소 중 dump_signal=1:    {df[df['is_korea_target']==0]['dump_signal'].sum()}건 "
      f"({df[df['is_korea_target']==0]['dump_signal'].mean()*100:.1f}%)")


dump_signal 분포:
dump_signal
0    39167
1    13191
Name: count, dtype: int64
한국 피소 중 dump_signal=1: 10479건 (39.3%)
비피소 중 dump_signal=1:    2712건 (10.6%)


### 6. 불필요 컬럼 제거

In [37]:
# ── Step 7. 불필요 컬럼 제거
drop_cols = [
    'country',           # imposing_country와 중복
    'hs_code',           # hs_4digit과 중복
    'target_country',    # 레이블 생성에 썼고 텍스트
    'tariff_clean',      # 텍스트, ML 입력 불가
    'start_date',        # duration_months로 대체
    'end_date',          # duration_months로 대체
    'date_type',         # 텍스트
    'year_month',        # start_ym으로 대체
    'imposing_country',  # imposing_country_encoded로 대체
    'hs_4digit',         # hs_encoded로 대체
    'regulation_type',   # regulation_type_encoded로 대체
    'regulation_status', # regulation_status_encoded로 대체
    'case_id',           # 식별자
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f"\n컬럼 제거 후: {df.columns.tolist()}")


컬럼 제거 후: ['export_value', 'export_weight', 'unit_price', 'months_before', 'start_ym', 'is_korea_target', 'price_chg_3m_new', 'price_chg_6m_new', 'price_chg_12m_new', 'vol_chg_3m_new', 'vol_chg_6m_new', 'vol_chg_12m_new', 'regulation_type_encoded', 'regulation_status_encoded', 'duration_months', 'imposing_country_encoded', 'hs_encoded', 'dump_signal']


### 7. 변화율 결측행 제거

In [38]:
chg_cols = ['price_chg_3m_new', 'vol_chg_3m_new',
            'price_chg_6m_new', 'vol_chg_6m_new',
            'price_chg_12m_new','vol_chg_12m_new']
before = len(df)
df = df.dropna(subset=chg_cols)
print(f"\n변화율 결측 제거: {before - len(df)}행 제거")
print(f"최종 행 수: {len(df)}행")


변화율 결측 제거: 3266행 제거
최종 행 수: 49092행


### 8. 최종 확인

In [39]:
print(f"\n최종 컬럼: {df.columns.tolist()}")
print(f"\nis_korea_target 분포:")
print(df['is_korea_target'].value_counts())
print(f"클래스 비율 1:{df['is_korea_target'].eq(0).sum()/df['is_korea_target'].sum():.1f}")
print(f"\n결측값:")
print(df.isnull().sum())


최종 컬럼: ['export_value', 'export_weight', 'unit_price', 'months_before', 'start_ym', 'is_korea_target', 'price_chg_3m_new', 'price_chg_6m_new', 'price_chg_12m_new', 'vol_chg_3m_new', 'vol_chg_6m_new', 'vol_chg_12m_new', 'regulation_type_encoded', 'regulation_status_encoded', 'duration_months', 'imposing_country_encoded', 'hs_encoded', 'dump_signal']

is_korea_target 분포:
is_korea_target
1    26024
0    23068
Name: count, dtype: int64
클래스 비율 1:0.9

결측값:
export_value                 0
export_weight                0
unit_price                   0
months_before                0
start_ym                     0
is_korea_target              0
price_chg_3m_new             0
price_chg_6m_new             0
price_chg_12m_new            0
vol_chg_3m_new               0
vol_chg_6m_new               0
vol_chg_12m_new              0
regulation_type_encoded      0
regulation_status_encoded    0
duration_months              0
imposing_country_encoded     0
hs_encoded                   0
dump_signal      

In [40]:
df.to_csv('ml_ready.csv', index=False, encoding='utf-8-sig')
print("\n✅ ml_ready.csv 저장 완료")


✅ ml_ready.csv 저장 완료
